# Experiment: MediaPipe Pose Demo

What this notebook teaches:
- Install and run MediaPipe Pose in a clean Colab runtime.
- Convert BlazePose landmarks into a shared canonical COCO-17 schema.
- Export frame-level keypoints to CSV and JSON with stable columns.
- Save visual artifacts for README usage.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


In [ ]:
def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("mediapipe", "opencv-python-headless", "matplotlib", "numpy", "pandas")


## Sample input

We download a small public-domain walking image and run one-frame inference.


In [ ]:
import urllib.request

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np

from posebench.export import export_frames_to_csv, export_frames_to_json
from posebench.keypoints_schema import map_tool_keypoints_to_canonical
from posebench.viz import draw_skeleton

sample_url = "https://upload.wikimedia.org/wikipedia/commons/9/92/Walking_person.jpg"
sample_path = repo_root / "assets" / "sample_input_walking.jpg"
sample_path.parent.mkdir(parents=True, exist_ok=True)

if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

bgr = cv2.imread(str(sample_path))
if bgr is None:
    raise RuntimeError("Could not read sample image")
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

pose = mp.solutions.pose.Pose(
    static_image_mode=True,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
)
result = pose.process(rgb)

if not result.pose_landmarks:
    raise RuntimeError("MediaPipe did not detect landmarks on sample image")

h, w = rgb.shape[:2]
points = []
for landmark in result.pose_landmarks.landmark:
    points.append(
        {
            "x": float(landmark.x * w),
            "y": float(landmark.y * h),
            "confidence": float(landmark.visibility),
        }
    )

canonical = map_tool_keypoints_to_canonical("mediapipe", points, min_confidence=0.1)
frame_payload = [
    {
        "frame_index": 0,
        "timestamp_ms": 0.0,
        "person_id": 0,
        "tool": "mediapipe",
        "schema": "coco17",
        "keypoints": canonical,
    }
]

export_frames_to_csv(frame_payload, repo_root / "results" / "mediapipe_demo_keypoints.csv")
export_frames_to_json(frame_payload, repo_root / "results" / "mediapipe_demo_keypoints.json")

rendered = draw_skeleton(bgr, canonical, min_confidence=0.1)
render_path = repo_root / "assets" / "generated" / "mediapipe_demo_overlay.jpg"
render_path.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(render_path), rendered)

plt.figure(figsize=(7, 4))
plt.imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("MediaPipe demo overlay")
plt.show()

print("Saved:")
print("-", repo_root / "results" / "mediapipe_demo_keypoints.csv")
print("-", repo_root / "results" / "mediapipe_demo_keypoints.json")
print("-", render_path)


## Mini benchmark

This short run measures **inference only** on a synthetic frame and stores raw JSON under `results/`.


In [ ]:
import numpy as np

from posebench.benchmark import BenchmarkConfig, benchmark_backend, write_json

class MPBackend:
    name = "mediapipe"

    def __init__(self, pose_model):
        self.pose_model = pose_model

    def infer(self, frame: np.ndarray):
        return self.pose_model.process(frame)

bench_frame = np.random.default_rng(7).integers(0, 255, size=(480, 640, 3), dtype=np.uint8)
bench_result = benchmark_backend(
    backend=MPBackend(pose),
    frames=[bench_frame],
    config=BenchmarkConfig(warmup_frames=8, measured_frames=32, repeat=1),
)

bench_path = repo_root / "results" / "benchmark_raw_mediapipe_demo.json"
write_json(bench_result, bench_path)
bench_result
